# Quantum Commerce — Treinando um modelo pequeno para resumir avaliações (PTT5-small)

**O que você vai fazer aqui:** treinar um modelo pequeno (PTT5-small, 60 milhões de parâmetros) para escrever o resumo em prosa das avaliações de um produto — aquele parágrafo estilo Mercado Livre ("As capas de almofada apresentam um visual atraente... Contudo, há críticas quanto à fragilidade..."). O professor desse modelo é o **Gemini**: ele escreve os resumos de treino, e o modelo pequeno aprende a imitá-lo. Isso se chama **destilação**.

**Onde isso entra no trabalho — leia antes de rodar:** este notebook é um experimento de **evolução futura / comparação** (tópico 4 do enunciado), **não** o modelo principal do trabalho. O núcleo do trabalho continua sendo o classificador de sentimento (TF-IDF + Regressão Logística, com a MLP como comparação) — o enunciado limita o núcleo até perceptron/MLP, e Transformer/generativo é evolução futura. A loja hoje usa o resumo do Gemini direto, em cache; aqui você testa se um modelo pequeno treinado por você consegue substituí-lo — se sim, ótimo; se não, a comparação entre os dois já é um resultado válido para o trabalho.

**O que você vai aprender, passo a passo:**

1. Montar grupos de avaliações por produto a partir do Olist
2. Gerar os dados de treino com o Gemini (destilação), usando o mesmo prompt do backend
3. Separar treino e validação
4. Fazer fine-tuning de um modelo seq2seq pequeno (PTT5-small) com o Hugging Face Trainer
5. Avaliar com ROUGE, comparar com o baseline determinístico e com o Gemini
6. Checar alucinação na mão e aplicar a regra de decisão (trocar ou não trocar)

**Como rodar:** no Colab, **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**, depois **Ambiente de execução → Executar tudo**. O fine-tuning precisa da GPU; na CPU levaria horas.

**Tempo estimado:** preparação ~2 min · destilação (500 grupos) ~20–40 min · fine-tuning ~10–20 min · avaliação ~2 min. A destilação é a parte longa: deixe rodando — ela salva o progresso e pode ser retomada se cair.

**De onde vêm os dados:** `olist_order_reviews_dataset.csv` + `olist_order_items_dataset.csv`, na pasta `TRABALHO-FINAL/archive/` do seu Drive. Nada precisa ser baixado.

**Resultado esperado (para conferir no final):** uma tabela ROUGE-1/2/L do modelo pequeno contra os resumos do Gemini, o ROUGE do baseline determinístico (o piso a superar) e a decisão — trocar ou manter o Gemini — com justificativa.


## Passo 0 — O ambiente: o que precisa estar instalado

O Colab já traz `pandas` e `torch`, mas as bibliotecas de NLP precisam ser instaladas: `transformers` (o modelo e o treino), `accelerate` (o `Trainer` usa por baixo), `rouge-score` (a avaliação) e `sentencepiece` (o tokenizador do T5 é baseado nele).

A célula abaixo instala tudo e confere a GPU. **Se `GPU disponível: False`, pare aqui** e ative a GPU T4 no menu **Ambiente de execução → Alterar o tipo de ambiente de execução**. Sem GPU o fine-tuning é inviável neste notebook.


In [ ]:
!pip install -q transformers accelerate rouge-score sentencepiece

import torch
print("torch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Passo 1 — Montar o Google Drive e apontar para os arquivos

O Colab roda numa máquina do Google, que não enxerga o seu Drive sozinha. A célula abaixo pede a sua autorização e monta o Drive em `/content/drive`. São quatro arquivos desta vez — além das avaliações, precisamos ligar cada avaliação ao produto dela (`order_items`) e traduzir a categoria (`products` + `product_category_name_translation`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA = '/content/drive/MyDrive/TRABALHO-FINAL/archive/'
ARQUIVOS = {
    'avaliacoes': 'olist_order_reviews_dataset.csv',
    'itens': 'olist_order_items_dataset.csv',
    'produtos': 'olist_products_dataset.csv',
    'categorias': 'product_category_name_translation.csv',
}
for nome, arquivo in ARQUIVOS.items():
    caminho = PASTA + arquivo
    print(f"{nome:12s} {'OK' if os.path.exists(caminho) else '⚠️ NÃO ENCONTRADO'}  {arquivo}")


## Passo 2 — Carregar os CSVs e entender a ligação avaliação → pedido → produto

O arquivo de avaliações **não tem o produto**: ele tem `order_id` (o pedido). Quem liga pedido a produto é o `order_items` — cada linha é um item de um pedido. Dois cuidados:

- **Pedidos com mais de um item:** 9.803 pedidos têm 2+ produtos. Nesses casos não dá para saber a qual produto a avaliação se refere — o cliente pode estar falando do outro item. Por isso este notebook usa **só pedidos de item único**: a avaliação pertence com certeza àquele produto.
- **Avaliações sem texto** não servem para resumir: são descartadas, como no treino de sentimento.


In [ ]:
import pandas as pd

avaliacoes = pd.read_csv(PASTA + ARQUIVOS['avaliacoes'])
itens = pd.read_csv(PASTA + ARQUIVOS['itens'], usecols=['order_id', 'product_id'])
produtos = pd.read_csv(PASTA + ARQUIVOS['produtos'], usecols=['product_id', 'product_category_name'])
categorias = pd.read_csv(PASTA + ARQUIVOS['categorias'])

print("Avaliações:", len(avaliacoes), "| Itens de pedido:", len(itens))
print("Produtos distintos no Olist:", itens['product_id'].nunique())
print("Pedidos com mais de 1 item:", (itens.groupby('order_id').size() > 1).sum())
print("Avaliações cujo pedido não está em itens:", (~avaliacoes['order_id'].isin(set(itens['order_id']))).sum())
avaliacoes.head(3)


## Passo 3 — Montar os grupos de avaliações por produto

Um "grupo" é tudo que o modelo vai ler de uma vez: as avaliações de **um** produto. As regras são as mesmas do prompt do backend (`backend/app/summary_prompt.py`):

- **Texto = título + mensagem**, como no treino de sentimento.
- **Só pedidos de item único** (Passo 2) e **só produtos com 3 ou mais avaliações com texto** — com menos de 3, nem o Gemini gera resumo na loja (`MIN_REVIEWS_FOR_AI_SUMMARY = 3`).
- **Ordem: da mais recente para a mais antiga**, e **no máximo 12 avaliações** por grupo (o mesmo teto do prompt do backend).
- **Nome do produto:** o Olist não traz nome de produto, só a categoria — então o grupo é identificado pela categoria traduzida (ex.: "beleza e saúde"). O resumo não depende do nome: o prompt proíbe inventar atributos.

Os números reais, medidos no Olist: **2.923 produtos** têm 3+ avaliações com texto em pedidos de item único, com **mediana de 4 avaliações por produto**. Vamos sortear **500** deles (semente 42). E um aviso honesto: o texto das avaliações é curto — **mediana de 52 caracteres** — então a entrada do modelo é fina. O resumo do Mercado Livre que você viu vem de milhares de comentários; aqui são 3 a 12 comentários curtos. Isso limita o que qualquer modelo consegue escrever.


In [ ]:
# Texto = título + mensagem (mesma regra do treino de sentimento).
avaliacoes['texto'] = (
    avaliacoes['review_comment_title'].fillna('') + ' ' +
    avaliacoes['review_comment_message'].fillna('')
).str.strip()
avaliacoes['data'] = pd.to_datetime(avaliacoes['review_creation_date'])

# Só pedidos de item único: a avaliação pertence com certeza àquele produto.
itens_por_pedido = itens.groupby('order_id').size()
pedidos_item_unico = set(itens_por_pedido[itens_por_pedido == 1].index)
com_texto = avaliacoes[(avaliacoes['texto'] != '') & (avaliacoes['order_id'].isin(pedidos_item_unico))]

# Liga avaliação -> produto e traduz a categoria.
ligadas = com_texto.merge(itens, on='order_id', how='inner')
ligadas = ligadas.merge(produtos, on='product_id', how='left')
ligadas = ligadas.merge(categorias, on='product_category_name', how='left')
ligadas['product_category_name_english'] = ligadas['product_category_name_english'].fillna('categoria não informada')

# Monta os grupos: avaliações ordenadas da mais recente para a mais antiga, teto de 12.
grupos = []
for product_id, g in ligadas.groupby('product_id'):
    g = g.sort_values('data', ascending=False).head(12)
    if len(g) < 3:
        continue
    grupos.append({
        'product_id': product_id,
        'categoria_pt': g['product_category_name_english'].iloc[0],
        'n_avaliacoes': len(g),
        'avaliacoes': [
            {'estrelas': int(r.review_score), 'texto': r.texto, 'data': str(r.data.date())}
            for r in g.itertuples()
        ],
    })

print("Produtos com 3+ avaliações (pedidos de item único):", len(grupos))
print("Mediana de avaliações por produto:", pd.Series([g['n_avaliacoes'] for g in grupos]).median())

# Sorteio dos 500 grupos do experimento (semente fixa = reproduzível).
import random
random.seed(42)
amostra = random.sample(grupos, 500)
print("Grupos sorteados para o experimento:", len(amostra))

# Um exemplo, para ver como o modelo vai receber os dados.
exemplo = amostra[0]
print("\nExemplo — categoria:", exemplo['categoria_pt'], "| avaliações:", exemplo['n_avaliacoes'])
for av in exemplo['avaliacoes']:
    print(f"  [{av['estrelas']} estrelas] {av['texto'][:120]}")


## Passo 4 — O professor: gerar os resumos com o Gemini (destilação)

Não existe dataset pronto em português com pares "avaliações → resumo". Então os dados de treino são criados aqui: o **Gemini escreve um resumo para cada grupo**, e o modelo pequeno aprende a imitar esses resumos. Isso se chama **destilação** — o modelo grande (professor) transfere o comportamento para o modelo pequeno (aluno).

Três consequências honestas, antes de rodar:

- **O teto de qualidade é o professor.** Se o Gemini errar ou inventar algo num resumo, o aluno aprende o erro junto. A destilação não cria qualidade nova — ela copia a que existe.
- **O prompt é exatamente o do backend** (`backend/app/summary_prompt.py`): as mesmas regras de fidelidade. A única diferença é o nome do produto — o Olist não traz nomes, então usamos a categoria traduzida. Se o prompt mudar lá, mude aqui também — senão o aluno aprende com regras diferentes das que a loja usa.
- **A chave nunca vai para o código.** Ela fica nos **segredos do Colab** (ícone 🔑 na barra lateral → "Adicionar novo segredo" → nome `GEMINI_API_KEY` → cole a chave) e é lida com `google.colab.userdata.get('GEMINI_API_KEY')`. O notebook nunca imprime a chave.

Como a chamada funciona: uma chamada REST por grupo (modelo `gemini-3.8-flash` — o `gemini-2.5-flash` dá 404 para chaves novas —, `temperature 0.2`, `maxOutputTokens 300`, os mesmos parâmetros do `backend/app/gemini_client.py`). Em caso de 429/503, espera 20 segundos e tenta **uma vez** de novo. Entre as chamadas, uma pausa de 1 segundo para respeitar o limite gratuito. **A cada 25 grupos, o progresso é salvo** num CSV no Drive — se a execução cair no meio, rode a célula de novo e ela continua de onde parou (grupos sem resumo são tentados de novo).


In [ ]:
import json, time
import requests
from google.colab import userdata

SAIDA = '/content/drive/MyDrive/TRABALHO-FINAL/modelo-resumo/'
os.makedirs(SAIDA, exist_ok=True)
CHECKPOINT = SAIDA + 'pares_checkpoint.csv'

# --- O prompt: IDÊNTICO a backend/app/summary_prompt.py ---
# Se mudar lá, mude aqui: o aluno precisa aprender com as mesmas regras da loja.
# (Só o PRODUTO muda: o Olist não traz nome de produto, então vai a categoria.)
def montar_prompt(nome_produto, avaliacoes):
    linhas = "\n".join(
        f"{i}. [{av['estrelas']} estrelas] {av['texto']}"
        for i, av in enumerate(avaliacoes, start=1)
    )
    return f"""Você resume avaliações de clientes de uma loja online, em português brasileiro.

PRODUTO: {nome_produto}

AVALIAÇÕES (nota de 1 a 5 + texto):
{linhas}

REGRAS (obrigatórias):
- Escreva um parágrafo de 2 a 4 frases, em português brasileiro.
- Afirme SOMENTE o que está escrito nas avaliações acima. Não invente atributos, materiais, preços, prazos, defeitos ou elogios que não apareçam nelas.
- Comece pelos pontos positivos e, se houver críticas, apresente-as depois com "Contudo," ou "Por outro lado,".
- Se não houver críticas nas avaliações, não invente nenhuma.
- Se as avaliações se contradizem, diga isso (por exemplo: "as opiniões divergem sobre...").
- Não cite nomes de clientes nem copie frases inteiras: resuma com suas palavras.
- Não mencione quantidades de avaliações nem notas.

Escreva apenas o parágrafo do resumo, sem título e sem explicações:"""

# --- A chamada ao Gemini: os mesmos parâmetros do backend/app/gemini_client.py ---
MODELO_PROFESSOR = 'gemini-3.8-flash'
CHAVE = userdata.get('GEMINI_API_KEY')
if not CHAVE:
    raise ValueError("Chave não encontrada: adicione o segredo GEMINI_API_KEY no Colab (ícone 🔑 na barra lateral).")

def gerar_resumo(prompt):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODELO_PROFESSOR}:generateContent"
    corpo = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.2, "maxOutputTokens": 300},
    }
    for tentativa in range(2):  # 1ª tentativa + 1 repetição em caso de 429/503
        resposta = requests.post(url, headers={"x-goog-api-key": CHAVE}, json=corpo, timeout=120)
        if resposta.status_code == 200:
            partes = resposta.json()["candidates"][0]["content"]["parts"]
            texto = "".join(p.get("text", "") for p in partes).strip()
            return texto or None
        if resposta.status_code in (429, 503) and tentativa == 0:
            time.sleep(20)
            continue
        return None
    return None

# --- Retoma de onde parou: só contam os grupos que já têm resumo ---
pares = []
if os.path.exists(CHECKPOINT):
    anteriores = pd.read_csv(CHECKPOINT).to_dict('records')
    for r in anteriores:
        if isinstance(r.get('resumo_gemini'), str) and r['resumo_gemini'].strip():
            pares.append(r)
    print(f"Checkpoint encontrado: {len(pares)} grupos já resumidos — serão pulados; falhas anteriores serão tentadas de novo.")
feitos = set(r['product_id'] for r in pares)

falhas = 0
for grupo in amostra:
    if grupo['product_id'] in feitos:
        continue
    nome = f"Produto da categoria '{grupo['categoria_pt']}'"
    resumo = gerar_resumo(montar_prompt(nome, grupo['avaliacoes']))
    if resumo is None:
        falhas += 1
    pares.append({
        'product_id': grupo['product_id'],
        'categoria_pt': grupo['categoria_pt'],
        'n_avaliacoes': grupo['n_avaliacoes'],
        'avaliacoes_json': json.dumps(grupo['avaliacoes'], ensure_ascii=False),
        'resumo_gemini': resumo,
    })
    time.sleep(1)  # respeita o limite gratuito da API
    if len(pares) % 25 == 0:
        pd.DataFrame(pares).to_csv(CHECKPOINT, index=False)
        print(f"... {len(pares)}/500 grupos processados")

pd.DataFrame(pares).to_csv(CHECKPOINT, index=False)
print(f"\nGrupos com resumo: {len(pares)} | falhas (sem resumo): {falhas}")
print("Arquivo salvo em:", CHECKPOINT)


## Passo 5 — Separar treino e validação (450/50)

A mesma disciplina do treino de sentimento: **a validação nunca é usada para treinar**. São 50 grupos guardados para medir, no final, se o modelo aprendeu ou só decorou. Grupos sem resumo (falhas da destilação) são descartados aqui — não há alvo para treinar.

Com 500 grupos e 50 de validação, sobram 450 para o treino. É pouco — e é parte do experimento: dá para ensinar um modelo pequeno a resumir com 450 exemplos? A resposta está no Passo 7.


In [ ]:
from sklearn.model_selection import train_test_split

pares_df = pd.read_csv(CHECKPOINT)
print("Grupos com resumo do Gemini:", pares_df['resumo_gemini'].notna().sum(), "de", len(pares_df))
pares_df = pares_df[pares_df['resumo_gemini'].notna()].reset_index(drop=True)

treino_df, val_df = train_test_split(pares_df, test_size=50, random_state=42)
print("Treino:", len(treino_df), "| Validação:", len(val_df))

treino_df.to_csv(SAIDA + 'pares_treino.csv', index=False)
val_df.to_csv(SAIDA + 'pares_val.csv', index=False)
print("Salvos em:", SAIDA)


## Passo 6 — O modelo aluno: PTT5-small (e o que ele pode e não pode fazer)

Resumir é uma tarefa de **texto entra, texto sai** — por isso o modelo é um **seq2seq** (codificador + decodificador): o codificador lê as avaliações e o decodificador escreve o parágrafo. É a mesma família do T5, e bem diferente do classificador do trabalho (que só escolhe entre 3 rótulos).

- **PTT5-small** (`unicamp-dl/ptt5-small`): um T5 de **60 milhões de parâmetros**, pré-treinado do zero em português brasileiro (corpus BrWaC, Unicamp). Ele já "sabe português" — o fine-tuning só ensina a tarefa (resumir avaliações neste formato). Se o download falhar, o código tenta o **mT5-small** (`google/mt5-small`, 300M, multilíngue) como reserva.
- **Por que pequeno?** Cabe na GPU gratuita do Colab (T4), treina em minutos e, depois, roda até em CPU para gerar um resumo. O preço é a capacidade: com 60M de parâmetros e 450 exemplos, ele **imita o estilo** do professor — não "entende" as avaliações.
- **O que ele não pode fazer:** inventar informação que não está nos dados (quando inventa, é alucinação — ver Passo 8), nem superar o professor. Se o resumo do Gemini de um grupo for fraco, o aluno aprende a versão fraca.

**O formato de entrada:** o T5 é um modelo *text-to-text* — toda tarefa começa com um prefixo que diz o que fazer. Aqui é `"resuma: "` seguido das avaliações separadas por ` | `. A saída é o resumo do Gemini.

**Detalhe técnico que decide se o treino funciona:** na tokenização, a entrada é cortada em **512 tokens** e o resumo em **128 tokens** (os grupos são curtos — cabem com folga). As posições de preenchimento (*padding*) do alvo são marcadas com **-100**, para a função de perda ignorá-las: o modelo só aprende com as palavras reais do resumo, não com o enchimento. O `DataCollatorForSeq2Seq` faz isso automaticamente.

**Hiperparâmetros:** 3 épocas, taxa de aprendizado 3e-4, batch 8, avaliação ao final de cada época (o melhor checkpoint é o que fica). São valores padrão para fine-tuning de T5-small — pequenos o bastante para 450 exemplos não decorarem tudo na primeira época.


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    DataCollatorForSeq2Seq, Trainer, TrainingArguments,
)

MODELO_BASE = 'unicamp-dl/ptt5-small'   # reserva: 'google/mt5-small'
PREFIXO = 'resuma: '
MAX_ENTRADA = 512
MAX_SAIDA = 128

try:
    tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
    modelo = AutoModelForSeq2SeqLM.from_pretrained(MODELO_BASE)
    print("Modelo base carregado:", MODELO_BASE)
except Exception as erro:
    print("Falha ao carregar", MODELO_BASE, "— tentando a reserva. Erro:", erro)
    MODELO_BASE = 'google/mt5-small'
    tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
    modelo = AutoModelForSeq2SeqLM.from_pretrained(MODELO_BASE)
    print("Modelo base carregado:", MODELO_BASE)

def montar_entrada(avaliacoes_json):
    avaliacoes = json.loads(avaliacoes_json)
    textos = " | ".join(f"[{av['estrelas']} estrelas] {av['texto']}" for av in avaliacoes)
    return PREFIXO + textos

class ParesResumo(Dataset):
    """Um item = (entrada tokenizada, alvo tokenizado). O collator faz o padding."""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        linha = self.df.iloc[i]
        entrada = tokenizer(montar_entrada(linha['avaliacoes_json']),
                            truncation=True, max_length=MAX_ENTRADA)
        alvo = tokenizer(linha['resumo_gemini'],
                         truncation=True, max_length=MAX_SAIDA)
        return {
            'input_ids': entrada['input_ids'],
            'attention_mask': entrada['attention_mask'],
            'labels': alvo['input_ids'],   # o collator troca o padding por -100
        }

treino_ds = ParesResumo(treino_df)
val_ds = ParesResumo(val_df)
print("Exemplo de entrada (primeiros 200 caracteres):")
print(montar_entrada(treino_df['avaliacoes_json'].iloc[0])[:200], "...")
print("Exemplo de alvo:", treino_df['resumo_gemini'].iloc[0][:200], "...")

collator = DataCollatorForSeq2Seq(tokenizer, model=modelo, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir='./modelo-resumo-ptt5',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    logging_steps=20,
    eval_strategy='epoch',          # avalia ao final de cada época (versões antigas: evaluation_strategy)
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,    # fica com o melhor checkpoint
    metric_for_best_model='eval_loss',
    report_to=[],
    seed=42,
    fp16=True,                      # T4 suporta; acelera o treino
)

trainer = Trainer(model=modelo, args=args, train_dataset=treino_ds,
                  eval_dataset=val_ds, data_collator=collator)
trainer.train()

trainer.save_model('./modelo-resumo-ptt5-final')
tokenizer.save_pretrained('./modelo-resumo-ptt5-final')
print("\nModelo salvo em ./modelo-resumo-ptt5-final")


## Passo 7 — Avaliar: ROUGE, e o que ele mede (e o que não mede)

**ROUGE** compara o resumo gerado com um resumo de referência, contando palavras em comum:

- **ROUGE-1:** palavras isoladas em comum. **ROUGE-2:** pares de palavras em comum (captura um pouco da fluência). **ROUGE-L:** a maior sequência de palavras em comum, em ordem.
- A referência aqui é o **resumo do Gemini**. Então o ROUGE mede **imitação do professor** — não qualidade absoluta. Um ROUGE-L de 0,5 não quer dizer "o resumo está 50% bom"; quer dizer "metade das palavras/sequências coincide com a do professor".
- Nem o próprio professor faria 1,0: com `temperature 0.2`, rodar o Gemini duas vezes no mesmo grupo dá textos diferentes. O teto prático está abaixo de 1,0 — e ninguém sabe exatamente onde, sem medir.
- **O ROUGE não enxerga alucinação.** Um resumo fluente que inventa um defeito pode ter ROUGE alto. Por isso o Passo 8 existe — e a decisão final nunca é só o número.

O `use_stemmer=False` é proposital: o stemmer padrão do ROUGE é inglês e estragaria as palavras em português.


In [ ]:
from rouge_score import RougeScorer

# Geração: busca em feixe (beam search) com 4 feixes — o modelo escolhe
# a sequência mais provável no conjunto, em vez de ir palavra por palavra.
modelo.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo.to(device)

def gerar_resumo_ptt5(avaliacoes_json):
    entrada = tokenizer(montar_entrada(avaliacoes_json), truncation=True,
                        max_length=MAX_ENTRADA, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = modelo.generate(**entrada, max_length=MAX_SAIDA, num_beams=4,
                              no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

val_df = val_df.copy()
val_df['resumo_ptt5'] = [gerar_resumo_ptt5(j) for j in val_df['avaliacoes_json']]
print("Resumos gerados:", len(val_df))

scorer = RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
def media_rouge(referencias, gerados):
    totais = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    for ref, ger in zip(referencias, gerados):
        pontos = scorer.score(ref, ger)
        for chave in totais:
            totais[chave] += pontos[chave].fmeasure
    return {chave: round(valor / len(referencias), 4) for chave, valor in totais.items()}

rouge_ptt5 = media_rouge(val_df['resumo_gemini'], val_df['resumo_ptt5'])
print("ROUGE do PTT5-small vs resumos do Gemini:", rouge_ptt5)


## Passo 7 (continuação) — O baseline determinístico: o piso a superar

O modelo pequeno precisa ganhar de **alguma coisa** para valer a pena — e o "alguma coisa" é o resumo determinístico que a loja já tem: contagem de sentimentos + termos mais citados, sem nenhuma palavra inventada. Ele é o piso: se o PTT5 não superar o ROUGE dele, o fine-tuning não trouxe nada.

A função abaixo é a versão em Python das mesmas regras do `deterministic_summary` do backend (`backend/app/services.py`) e do `buildReviewSummary` do front — distribuição por sentimento (1–2 = negativa, 3 = neutra, 4–5 = positiva) e os 3 termos mais citados de cada lado, com uma lista reduzida de stopwords.


In [ ]:
import re
from collections import Counter

# Lista reduzida de stopwords — a completa vive em backend/ml/stopwords.py.
STOPWORDS = {'a', 'o', 'e', 'de', 'do', 'da', 'em', 'um', 'uma', 'que', 'com',
             'para', 'os', 'as', 'dos', 'das', 'se', 'não', 'na', 'no', 'por',
             'mais', 'muito', 'bem', 'foi', 'é', 'são', 'me', 'eu', 'ele', 'ela',
             'esse', 'essa', 'isto', 'isso', 'como', 'mas', 'ao', 'aos', 'ou',
             'já', 'só', 'também', 'quando', 'onde', 'seu', 'sua', 'meu', 'minha'}

def rotulo(estrelas):
    return 'negativa' if estrelas <= 2 else ('neutra' if estrelas == 3 else 'positiva')

def termos_mais_citados(avaliacoes, sentimento):
    palavras = []
    for av in avaliacoes:
        if rotulo(av['estrelas']) != sentimento:
            continue
        palavras += [p for p in re.findall(r'[a-zà-ú]{3,}', av['texto'].lower())
                     if p not in STOPWORDS]
    return [termo for termo, _ in Counter(palavras).most_common(3)]

def resumo_deterministico(avaliacoes):
    total = len(avaliacoes)
    contagem = Counter(rotulo(av['estrelas']) for av in avaliacoes)
    partes = [f"De {total} avaliações, {contagem.get('positiva', 0)} positivas, "
              f"{contagem.get('neutra', 0)} neutras e {contagem.get('negativa', 0)} negativas."]
    elogios = termos_mais_citados(avaliacoes, 'positiva')
    if elogios:
        partes.append(f"O que mais elogiam: {', '.join(elogios)}.")
    reclamacoes = termos_mais_citados(avaliacoes, 'negativa')
    if reclamacoes:
        partes.append(f"O que mais reclamam: {', '.join(reclamacoes)}.")
    return ' '.join(partes)

val_df['resumo_deterministico'] = [
    resumo_deterministico(json.loads(j)) for j in val_df['avaliacoes_json']
]
rouge_det = media_rouge(val_df['resumo_gemini'], val_df['resumo_deterministico'])

print("ROUGE-1 / ROUGE-2 / ROUGE-L (F1, referência = resumo do Gemini)")
print(f"  Determinístico (piso): {rouge_det}")
print(f"  PTT5-small (aluno):    {rouge_ptt5}")


## Passo 8 — Lado a lado e o checklist manual de alucinação

Número não lê. A célula abaixo imprime 10 exemplos da validação com tudo junto — as avaliações originais, o resumo do Gemini, o do PTT5 e o determinístico — para você ler e julgar. Para cada exemplo do PTT5, marque:

- [ ] **Inventou?** Algum atributo, material, defeito ou elogio que **não está** nas avaliações?
- [ ] **Omitiu a crítica principal?** Havia uma reclamação clara e o resumo não a mencionou?
- [ ] **Contradiz as avaliações?** Por exemplo: as avaliações dizem "não recomendo" e o resumo diz que recomendam, ou o tom (positivo/negativo) está invertido?
- [ ] **Trocou o sentido?** "Chegou rápido, mas quebrou" virando só "chegou rápido"?

Anote quantos dos 10 exemplos têm pelo menos um problema — esse número entra na regra de decisão do Passo 9.


In [ ]:
AMOSTRA_MANUAL = val_df.head(10)

for k, linha in enumerate(AMOSTRA_MANUAL.itertuples(), start=1):
    avaliacoes = json.loads(linha.avaliacoes_json)
    print(f"\n{'=' * 70}\nEXEMPLO {k} — categoria: {linha.categoria_pt} | {linha.n_avaliacoes} avaliações")
    for av in avaliacoes:
        print(f"  [{av['estrelas']}★] {av['texto']}")
    print(f"\n  Gemini (professor):     {linha.resumo_gemini}")
    print(f"  PTT5-small (aluno):     {linha.resumo_ptt5}")
    print(f"  Determinístico (piso):  {linha.resumo_deterministico}")


## Passo 9 — A regra de decisão: trocar ou manter o Gemini?

A troca só acontece se **todas** as condições forem verdadeiras:

1. **ROUGE-L do PTT5 ≥ 0,5** — perto o bastante do professor para a imitação ser real, não coincidência de palavras soltas;
2. **Zero alucinações nos 10 exemplos lidos** no Passo 8 — um modelo que inventa defeito não pode escrever o resumo da loja;
3. **A leitura lado a lado é aceitável** — os resumos do aluno soam como resumos, não como frases soltas.

Se qualquer uma falhar, **mantemos o Gemini** e o trabalho ganha a comparação documentada — um resultado negativo bem medido vale tanto quanto um positivo, e o enunciado pede justamente avaliação honesta (tópicos 2 e 4). O número 0,5 é um ponto de partida, não uma lei: se o ROUGE-L ficar em 0,48 com zero alucinações e resumos bons na leitura, a decisão é sua — mas registre o porquê.


In [ ]:
# Preencha após ler os 10 exemplos do Passo 8:
ALUCINACOES_ENCONTRADAS = 0   # <-- quantos dos 10 exemplos têm pelo menos um problema?

print("ROUGE-L do PTT5-small:", rouge_ptt5['rougeL'], "| piso determinístico:", rouge_det['rougeL'])
print("Alucinações na amostra manual (0–10):", ALUCINACOES_ENCONTRADAS)

trocar = rouge_ptt5['rougeL'] >= 0.5 and ALUCINACOES_ENCONTRADAS == 0
print("\nDECISÃO:", "TROCAR pelo PTT5-small — documentar a troca no plano e no README."
      if trocar else "MANTER o Gemini — registrar a comparação (ROUGE + leitura) no plano.")


## Passo 10 — Salvar o modelo e as métricas no Drive

O experimento só existe se estiver salvo: o modelo treinado (para carregar depois sem treinar de novo), os pares gerados e um `metrics.json` com os números e a decisão — o mesmo padrão do treino de sentimento.


In [ ]:
import json, shutil

SAIDA_MODELO = SAIDA + 'modelo-resumo-ptt5/'
os.makedirs(SAIDA_MODELO, exist_ok=True)

# O modelo final já foi salvo em ./modelo-resumo-ptt5-final — copia para o Drive.
shutil.copytree('./modelo-resumo-ptt5-final', SAIDA_MODELO, dirs_exist_ok=True)

metricas = {
    "modelo_base": MODELO_BASE,
    "modelo": "PTT5-small fine-tunado (destilação do Gemini)",
    "n_treino": int(len(treino_df)),
    "n_validacao": int(len(val_df)),
    "hiperparametros": {"epochs": 3, "lr": 3e-4, "batch": 8, "max_entrada_tokens": MAX_ENTRADA, "max_saida_tokens": MAX_SAIDA},
    "professor": {"modelo": MODELO_PROFESSOR, "temperature": 0.2, "max_output_tokens": 300},
    "rouge_vs_gemini": {"ptt5_small": rouge_ptt5, "deterministico": rouge_det},
    "alucinacoes_amostra_manual_10": ALUCINACOES_ENCONTRADAS,
    "decisao": "trocar" if trocar else "manter_gemini",
    "fonte": "olist_order_reviews_dataset.csv + olist_order_items_dataset.csv (Olist, 2016–2018)",
}
with open(SAIDA_MODELO + 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

print("Salvo em:", SAIDA_MODELO)
print(json.dumps(metricas, indent=2, ensure_ascii=False))


## Checklist — o que você deve conseguir explicar agora

- [ ] O que é destilação e por que o teto de qualidade do aluno é o professor
- [ ] Por que o dataset de pares não existe pronto e como ele foi criado
- [ ] Por que só pedidos de item único entram nos grupos
- [ ] O que o prefixo `"resuma: "` faz e por que a entrada é cortada em 512 tokens
- [ ] Para que serve o `-100` nas labels e o que o `DataCollatorForSeq2Seq` faz
- [ ] O que o ROUGE mede — e por que ele não enxerga alucinação
- [ ] Por que o baseline determinístico é o piso e o que acontece se o aluno não o superar
- [ ] Qual foi a decisão (trocar ou manter) e com quais números ela foi tomada

## Limites honestos deste experimento

- **A entrada é fina:** mediana de 4 avaliações por produto, com mediana de 52 caracteres cada. O resumo do Mercado Livre vem de milhares de comentários; aqui o modelo escreve a partir de 3 a 12 frases curtas. Não espere a mesma riqueza.
- **Destilação copia — inclusive os erros.** Se o professor inventou um atributo num resumo, o aluno aprende a inventar também. A checagem manual do Passo 8 existe por isso.
- **450 exemplos é pouco.** O modelo aprende o formato e o estilo; generalização real exigiria mais pares (gerar 1.000–2.000 grupos é o próximo passo óbvio, se o tempo permitir).
- **O PTT5-small é de 2020.** Alternativas mais novas: `ptt5-v2-small` (mesmo grupo, pré-treino continuado) ou o `ptt5-base` (220M, recomendado pelos autores, mas mais pesado para a GPU gratuita).
- **ROUGE mede imitação, não verdade.** Um ROUGE-L alto com alucinações não vale nada — a decisão final sempre passa pela leitura.

**Onde isso entra no projeto:** este experimento é a seção de evolução futura do plano (`docs/plano-trabalho-final.md`, §8) e a comparação do tópico 4. Se a decisão for trocar, a troca no backend seria um novo preditor implementando a mesma interface dos outros modelos (o padrão SOLID do projeto) — mas isso é um passo futuro, fora deste notebook.
